In [1]:
# =====================
# CELL 1: Install deps
# =====================
!pip install -q datasets pillow tqdm

In [2]:
# =====================
# CELL 2: Imports
# =====================
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm
import os
import random

In [3]:
BASE_DIR = "classification_data"
IMG_SIZE = (224, 224)
SPLITS = {"train": 0.7, "val": 0.15, "test": 0.15}

# 25 classes (same as YOLO)
COCO_CLASSES = {
    "person": 0,
    "bicycle": 1,
    "car": 2,
    "motorcycle": 3,
    "airplane": 4,
    "bus": 5,
    "truck": 7,
    "traffic light": 9,
    "stop sign": 11,
    "bench": 13,
    "bird": 14,
    "cat": 15,
    "dog": 16,
    "horse": 17,
    "cow": 19,
    "elephant": 20,
    "bottle": 39,
    "cup": 41,
    "bowl": 45,
    "chair": 56,
    "couch": 57,
    "potted plant": 58,
    "bed": 59,
    "pizza": 60,
    "cake": 61
}

ID_TO_CLASS = {v: k for k, v in COCO_CLASSES.items()}

# Create directory structure
for split in SPLITS:
    for cls in COCO_CLASSES:
        os.makedirs(os.path.join(BASE_DIR, split, cls), exist_ok=True)


In [4]:
# =====================
# CELL 4: Load COCO (streaming)
# =====================
print("Loading COCO dataset (streaming)...")
dataset = load_dataset("detection-datasets/coco", split="train", streaming=True)

Loading COCO dataset (streaming)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

In [5]:
from collections import defaultdict
from tqdm import tqdm

MAX_CROPS_PER_CLASS = 500
class_counter = defaultdict(int)
all_crops = []

print("Extracting object crops (limited per class)...")

for sample in tqdm(dataset):
    image = sample["image"].convert("RGB")
    boxes = sample["objects"]["bbox"]
    cats  = sample["objects"]["category"]

    for bbox, cid in zip(boxes, cats):
        if cid in ID_TO_CLASS and class_counter[cid] < MAX_CROPS_PER_CLASS:
            x, y, w, h = bbox
            crop = image.crop((int(x), int(y), int(x + w), int(y + h)))
            crop = crop.resize(IMG_SIZE)

            cls_name = ID_TO_CLASS[cid]
            all_crops.append((crop, cls_name))
            class_counter[cid] += 1

    if all(class_counter[cid] >= MAX_CROPS_PER_CLASS for cid in ID_TO_CLASS):
        break

print("Crops per class:")
for cid in ID_TO_CLASS:
    print(ID_TO_CLASS[cid], class_counter[cid])

print("Total crops collected:", len(all_crops))


Extracting object crops (limited per class)...


30584it [11:56, 42.70it/s] 

Crops per class:
person 500
bicycle 500
car 500
motorcycle 500
airplane 500
bus 500
truck 500
traffic light 500
stop sign 500
bench 500
bird 500
cat 500
dog 500
horse 500
cow 500
elephant 500
bottle 500
cup 500
bowl 500
chair 500
couch 500
potted plant 500
bed 500
pizza 500
cake 500
Total crops collected: 12500


In [6]:
# =====================
# CELL 6: Shuffle & split
# =====================
random.shuffle(all_crops)
N = len(all_crops)
train_end = int(N * SPLITS["train"])
val_end = train_end + int(N * SPLITS["val"])


train_data = all_crops[:train_end]
val_data = all_crops[train_end:val_end]
test_data = all_crops[val_end:]

In [8]:
# =====================
# CELL 7: Save crops to disk
# =====================
def save_split(data, split_name):
  counters = {}
  for img, cls in tqdm(data):
    counters.setdefault(cls, 0)
    fname = f"{cls}_{counters[cls]}.jpg"
    path = os.path.join(BASE_DIR, split_name, cls, fname)
    img.save(path)
    counters[cls] += 1


print("Saving training crops...")
save_split(train_data, "train")


print("Saving validation crops...")
save_split(val_data, "val")


print("Saving test crops...")
save_split(test_data, "test")

Saving training crops...


100%|██████████| 8750/8750 [00:04<00:00, 1926.27it/s]


Saving validation crops...


100%|██████████| 1875/1875 [00:01<00:00, 1847.79it/s]


Saving test crops...


100%|██████████| 1875/1875 [00:00<00:00, 1982.06it/s]


In [10]:
# =====================
# CELL 8: Sanity check
# =====================
print("\nSanity check (images per class in TRAIN):")
for cls in COCO_CLASSES:
  count = len(os.listdir(os.path.join(BASE_DIR, "train", cls)))
  print(cls, count)


print("\nClassification dataset preparation COMPLETE ✅")


Sanity check (images per class in TRAIN):
person 345
bicycle 345
car 353
motorcycle 350
airplane 346
bus 364
truck 368
traffic light 356
stop sign 348
bench 347
bird 344
cat 364
dog 362
horse 341
cow 343
elephant 339
bottle 354
cup 369
bowl 339
chair 342
couch 349
potted plant 351
bed 335
pizza 341
cake 355

Classification dataset preparation COMPLETE ✅


In [12]:
!zip -r class_sample.zip classification_data/train


Streaming output truncated to the last 5000 lines.
  adding: classification_data/train/dog/dog_140.jpg (deflated 1%)
  adding: classification_data/train/dog/dog_94.jpg (deflated 23%)
  adding: classification_data/train/dog/dog_304.jpg (deflated 23%)
  adding: classification_data/train/dog/dog_191.jpg (deflated 16%)
  adding: classification_data/train/dog/dog_125.jpg (deflated 2%)
  adding: classification_data/train/dog/dog_315.jpg (deflated 12%)
  adding: classification_data/train/dog/dog_349.jpg (deflated 4%)
  adding: classification_data/train/dog/dog_88.jpg (deflated 6%)
  adding: classification_data/train/dog/dog_184.jpg (deflated 4%)
  adding: classification_data/train/dog/dog_194.jpg (deflated 5%)
  adding: classification_data/train/dog/dog_216.jpg (deflated 1%)
  adding: classification_data/train/dog/dog_235.jpg (deflated 14%)
  adding: classification_data/train/dog/dog_84.jpg (deflated 15%)
  adding: classification_data/train/dog/dog_203.jpg (deflated 8%)
  adding: classificati

In [13]:
from google.colab import files
files.download("class_sample.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>